In [ ]:
'''
This cell sets up the Google Colab environment.
This allows other uses to use and test my code on their own devises
It defines the GitHub repository structure where the pre-trained model is stored.
The line urllib.request.urlretrieve(raw_model_url, MODEL_FILENAME) is responsible for downloading the model file from the internet.
It then constructs the raw GitHub URL for the model.
Finally, it defines and executes a function to download the 'dnn_classifier.keras' file
into the Colab environment, ensuring the model is available for use.
'''

import os
import urllib.request #Python module that provides functions and classes to open URLs (mostly HTTP) in a complex world
from tensorflow.keras.models import load_model

# 1. Configuration based  repository structure
GITHUB_USERNAME = "keden49"
REPO_NAME = "Machine-learning-with-python-freecode-camp"
SUB_FOLDER = "Iris-classifier-keras"
MODEL_FILENAME = "dnn_classifier.keras"

# Construct the Raw Github URL Location of model online
raw_model_url = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/{REPO_NAME}/main/{SUB_FOLDER}/{MODEL_FILENAME}"

def setup_colab_env():
    """Downloads the pre-trained model into the Colab environment."""
    if not os.path.exists(MODEL_FILENAME):
        print(f" Fetching {MODEL_FILENAME} from {SUB_FOLDER}...")
        try:
            urllib.request.urlretrieve(raw_model_url, MODEL_FILENAME)
            print(" Model successfully added to environment.")
        except Exception as e:
            print(f" Download failed. Check if {MODEL_FILENAME} is pushed to the {SUB_FOLDER} folder.")
            print(f"Error details: {e}")
    else:
        print(f" {MODEL_FILENAME} is ready.")

# Execute the setup
setup_colab_env()

 dnn_classifier.keras is ready.


In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals # Import future features for Python 2/3 compatibility
import tensorflow as tf # Import TensorFlow for machine learning
import pandas as pd # Import pandas for data manipulation and analysis

In [ ]:
CSV_COLUMN_NAMES= ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth', 'Species']
SPECIES= ['Setosa', 'Versicolor', 'Virginica']


In [ ]:
'''
 uses TensorFlow's utility function to download dataset files:
 The lines download the Iris training and testing dataset (a CSV file) from the specified Google Cloud Storage URL.
 The downloaded files will be saved locally as iris_training.csv and iris_testing.csv respectively, and its local path is stored in a variable.
'''

train_path = tf.keras.utils.get_file("iris_training.csv", "https://storage.googleapis.com/download.tensorflow.org/data/iris_training.csv")
test_path = tf.keras.utils.get_file("iris_test.csv", "https://storage.googleapis.com/download.tensorflow.org/data/iris_test.csv")

2194/2194 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
573/573 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step


In [ ]:
''''
Loading dataframes using pandas
pd.read_csv() is used to read the CSV files into pandas dataframes.
paths show the direction of files
names is for column names
header=0 takes row 0 as header
'''
train_df = pd.read_csv(train_path,names=CSV_COLUMN_NAMES,header=0)
test_df =pd.read_csv(test_path,names=CSV_COLUMN_NAMES,header=0)

In [ ]:
#visualizing first few columns of training ds
train_df.head()

,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
0,6.4,2.8,5.6,2.2,2
1,5.0,2.3,3.3,1.0,1
2,4.9,2.5,4.5,1.7,2
3,4.9,3.1,1.5,0.1,0
4,5.7,3.8,1.7,0.3,0


In [ ]:
#visualizing first few columns of testing ds
test_df.head()

,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
0,5.9,3.0,4.2,1.5,1
1,6.9,3.1,5.4,2.1,2
2,5.1,3.3,1.7,0.5,0
3,6.0,3.4,4.5,1.6,1
4,5.5,2.5,4.0,1.3,1


In [ ]:
#target labels
y_train =train_df.pop('Species')
y_test =test_df.pop('Species')

In [ ]:

'''
shape
120 rows
4 columns
'''

train_df.shape

(120, 4)

In [ ]:
'''
shape
30 rows
4 columns
'''

test_df.shape

(30, 4)

In [ ]:
'''
input shape of data
what changes is the number of rows
but what remains constant is the number of columns
'''
num_features = train_df.shape[1]

In [ ]:
'''
Building DNN Model
relu is the activation fn applied to the output of each neuron
outputs the input directly if it's positive, otherwise, it outputs zero.
ReLU allows the network to learn complex, non-linear patterns.
10 neurons in the 1st hidden layer learn more abstract patterns passed from the input layer
softmax is used for multi-class classification
Each of these 3 neurons will correspond to one of the target classes.
Softmax takes the raw output scores (logits) from the 3 neurons and squashes them into a probability distribution
sum will add up to 1
'''
dnn_classifier = tf.keras.Sequential([
    tf.keras.layers.Dense(30,activation = 'relu',input_shape=(num_features,)), #input layer
    tf.keras.layers.Dense(10,activation = 'relu'), #hidden Layer 1
    tf.keras.layers.Dense(3, activation='softmax') # Output layer with 3 units for 3 species and softmax activation
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
'''
configuring the model
optimizer is the algorithm that adjustes the classifiers internal parameters(weights and biases)
loss quantifies how far off the model's predictions are from the actual target values.
sparse categorical crossentropy is used when the target labels are intergers instead of hot encoded
Metrics are used to estimate model's perfomance
'''

dnn_classifier.compile(optimizer = 'adam',
                       loss = 'sparse_categorical_crossentropy',
                       metrics = ['accuracy'])

In [ ]:
'''
importing Early stopping to prevent overfitting
stops training when no improvement is seen in validation loss
patience tracks the no of epochs to tolerate before stopping
'''
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10,restore_best_weights=True,min_delta=0.00001)


In [ ]:
training_classifier = dnn_classifier.fit(train_df,y_train,
                                         epochs=100,
                                         batch_size=10,#15 weight updates per epoch
                                         validation_data=(test_df,y_test),#Separate data the model hasn't seen during training
                                         callbacks=[early_stop],verbose=1)

Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5038 - loss: 1.0869 - val_accuracy: 0.5333 - val_loss: 1.1226
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7218 - loss: 0.9145 - val_accuracy: 0.5333 - val_loss: 0.9954
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7594 - loss: 0.8207 - val_accuracy: 0.5333 - val_loss: 0.9427
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7494 - loss: 0.7585 - val_accuracy: 0.5333 - val_loss: 0.8990
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6713 - loss: 0.7918 - val_accuracy: 0.5333 - val_loss: 0.8651
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6960 - loss: 0.7203 - val_accuracy: 0.5333 - val_loss: 0.8424
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7390 - loss: 0.6652 - val_accuracy: 0.5333 - val_loss: 0.8165
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7018 - loss: 0.6508 - val_accuracy: 0.5333 -

In [ ]:
dnn_classifier.save('dnn_classifier.keras') #saves models last epoch (best weights)

In [ ]:
#using model and loading model
from tensorflow.keras.models import load_model
import numpy as np
loaded_model = load_model('dnn_classifier.keras')

In [ ]:
'''
creating user interface
GOAL : should collect data from user
'''
features = ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth']
flowers_info = []

print("Please enter as prompted\n")

for feature in features:
  # Loop until a valid numeric input is received for the current feature
  while True:
    val_str = input(feature + ": ") # Get input directly from user
    if not val_str.isdigit(): # Check if it's a valid number (integer or float)
      print("Invalid input. Please enter a numerical value.")
    # only if user input is valid is it converted to a floaat & appended
    else:
      val = float(val_str)
      flowers_info.append(val)
      break # Break out of the inner loop for this feature if valid

Please enter as prompted

SepalLength: 7
SepalWidth: 2
PetalLength: 1
PetalWidth: 1


In [ ]:
#unpacking users info
print(flowers_info)

[7.0, 2.0, 1.0, 1.0]


In [ ]:
'''
testing models predictions with users info
the model needs the data in a 2D array(row,column)
also needs to be converted to a numpy array
np.argmax returns the indices of the maximum values along an axis.
predictions[0] because it returns a list inside a list hence being the only item it takes index 0
to access the most probable specie i use class_ids
which is returned by np.argmax (index of highest value)
using that i can now get the probability which will in turn give me the specie
ouput is formated to 2 dp
'''
import numpy as np
user_input_array = np.array(flowers_info).reshape(1,-1)
predictions = loaded_model.predict(user_input_array)

#print(predictions)

class_ids= np.argmax(predictions[0])
probabilities = predictions[0][class_ids]
#print(probabilities)
print(f'The flower Specie is {SPECIES[class_ids]} with a probability of ({100 * probabilities:.2f}%)')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
The flower Specie is Setosa with a probability of (99.84%)
